# Minimal llmNER pipeline: multilingual zero-shot NER → BIO tags

**Install:**
```
pip install "git+https://github.com/plncmm/llmner.git"
brew install ollama   (then: ollama pull mistral && ollama serve)
```

**Usage:**
in a terminal pane:
```
ollama serve
```

To switch models, change `OLLAMA_MODEL` below.

To use OpenAI instead, set `USE_OLLAMA = False` and set `OPENAI_API_KEY`.

In [1]:
import os
from llmner import ZeroShotNer

# === Ollama config (free, local Large Language Model) ===
USE_OLLAMA = True
OLLAMA_MODEL = "ministral-3:14b"   # "ministral-3:14b" is a 14B-parameter model, good for NER tasks; see https://ollama.com/models for more options

if USE_OLLAMA:
    os.environ["OPENAI_API_KEY"] = "INNOVAFERTANIMUSMUTANTASDICEREFORMAS"     # placeholder  
    os.environ["OPENAI_API_BASE"] = "http://localhost:11434/v1"

In [2]:
# Input Texts & Entity Definitions
texts = [
    "My name is Wolfgang and I work at Google in Berlin.",          # English
    "Je m'appelle Marie et je travaille chez Airbus à Toulouse.",   # French
    "Mi chiamo Luigi e lavoro alla Ferrari a Maranello.",           # Italian
]

# TODO: Latin annotations have PERS, GRP, GEO
entities = {
    "person":       "A person's name, e.g. Wolfgang, Marie, Luigi",
    "organization": "A company or organisation name, e.g. Google, Airbus, Ferrari",
    "location":     "A city, country, or place name, e.g. Berlin, Toulouse, Maranello",
}

# Label abbreviation map
LABEL_SHORT = {
    "person":       "PER",
    "organization": "ORG",
    "location":     "LOC",
}


In [ ]:
# === Build, Contextualize ===
model = ZeroShotNer(model=OLLAMA_MODEL)          # pass model="gpt-4.1-nano" etc. if using API
model.contextualize(entities=entities)

# actually predict
annotated_docs = model.predict(texts)

  0%|          | 0/3 [00:00<?, ? example/s]Retrying langchain.chat_models.openai.ChatOpenAI.completion_with_retry.<locals>._completion_with_retry in 4.0 seconds as it raised APIConnectionError: Error communicating with OpenAI: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 61] Connection refused")).


In [ ]:
# === Helpers to convert output into proper labels ===
def spans_to_bio(text: str, annotations) -> list[tuple[str, str]]:
    """
    Tokenise `text` on whitespace and map character-level Annotation spans
    onto BIO labels.  Returns [(token, tag), ...].

    Annotation objects expose:  .start  .end  .label
    (end is exclusive, matching Python slice convention)
    """
    # Build a char → (label, is_begin) lookup from sorted, non-overlapping spans
    char_label: dict[int, tuple[str, bool]] = {}
    for ann in sorted(annotations, key=lambda a: a.start):
        short = LABEL_SHORT.get(ann.label, ann.label.upper())
        for i in range(ann.start, ann.end):
            char_label[i] = (short, i == ann.start)

    tokens: list[tuple[str, str]] = []
    cursor = 0
    for token in text.split():
        # Find where this token sits in the original string
        start = text.index(token, cursor)
        end = start + len(token)
        cursor = end

        # Majority-vote: take the label of the first annotated char in the token
        tag = "O"
        for i in range(start, end):
            if i in char_label:
                label, is_begin = char_label[i]
                tag = f"B-{label}" if is_begin else f"I-{label}"
                break

        tokens.append((token, tag))

    return tokens

In [ ]:
# print helpers
for text, doc in zip(texts, annotated_docs):
    print(f"\nText : {text}")
    print(f"Raw annotations : {doc.annotations}")

    bio = spans_to_bio(text, doc.annotations)
    print("BIO tags:")
    for token, tag in bio:
        print(f"  {token:<20} {tag}")

    # Compact list form (matches the format from your prompt)
    print("Compact:", [tag for _, tag in bio])


Text : My name is Wolfgang and I work at Google in Berlin.
Raw annotations : {Annotation(start=34, end=40, label='organization', text='Google'), Annotation(start=11, end=19, label='person', text='Wolfgang'), Annotation(start=44, end=50, label='location', text='Berlin')}
BIO tags:
  My                   O
  name                 O
  is                   O
  Wolfgang             B-PER
  and                  O
  I                    O
  work                 O
  at                   O
  Google               B-ORG
  in                   O
  Berlin.              B-LOC
Compact: ['O', 'O', 'O', 'B-PER', 'O', 'O', 'O', 'O', 'B-ORG', 'O', 'B-LOC']

Text : Je m'appelle Marie et je travaille chez Airbus à Toulouse.
Raw annotations : {Annotation(start=49, end=57, label='location', text='Toulouse'), Annotation(start=40, end=46, label='organization', text='Airbus'), Annotation(start=13, end=18, label='person', text='Marie')}
BIO tags:
  Je                   O
  m'appelle            O
  Marie         

In [ ]:
# same sample as Minimal_Pipeline_Demo

texts_latin =  ['Quousque tandem abutere, Catilina patientia nostra?']

annotated_docs_latin = model.predict(texts_latin)

for text, doc in zip(texts_latin, annotated_docs_latin):
    print(f"\nText : {text}")
    print(f"Raw annotations : {doc.annotations}")

    bio = spans_to_bio(text, doc.annotations)
    print("BIO tags:")
    for token, tag in bio:
        print(f"  {token:<20} {tag}")

    # Compact list form (matches the format from your prompt)
    print("Compact:", [tag for _, tag in bio])

100%|██████████| 1/1 [00:02<00:00,  2.03s/ example]


Text : Quousque tandem abutere, Catilina patientia nostra?
Raw annotations : set()
BIO tags:
  Quousque             O
  tandem               O
  abutere,             O
  Catilina             O
  patientia            O
  nostra?              O
Compact: ['O', 'O', 'O', 'O', 'O', 'O']
